## Interval Analysis

In [1]:
# !pip install tensorboardX

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import time
import matplotlib.pyplot as plt

from torchvision import datasets, transforms
# from tensorboardX import SummaryWriter
from torch.utils.data import DataLoader


use_cuda = False
device = torch.device("cuda" if use_cuda else "cpu")
batch_size = 64

np.random.seed(42)
torch.manual_seed(42)


## Dataloaders
train_dataset = datasets.MNIST('mnist_data/', train=True, download=True, transform=transforms.Compose(
    [transforms.ToTensor()]
))
test_dataset = datasets.MNIST('mnist_data/', train=False, download=True, transform=transforms.Compose(
    [transforms.ToTensor()]
))

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

## Simple NN. You can change this if you want. If you change it, mention the architectural details in your report.
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.fc = nn.Linear(28*28, 200)
        self.fc2 = nn.Linear(200,10)

    def forward(self, x):
        x = x.view((-1, 28*28))
        x = F.relu(self.fc(x))
        x = self.fc2(x)
        x = F.softmax(x, dim=-1) # added softmax for probabilities
        return x

class Normalize(nn.Module):
    def forward(self, x):
        return (x - 0.1307)/0.3081

# Add the data normalization as a first "layer" to the network
# this allows us to search for adverserial examples to the real image, rather than
# to the normalized image
model = nn.Sequential(Normalize(), Net())

model = model.to(device)
model.train()


Sequential(
  (0): Normalize()
  (1): Net(
    (fc): Linear(in_features=784, out_features=200, bias=True)
    (fc2): Linear(in_features=200, out_features=10, bias=True)
  )
)

In [2]:
def train_model(model, num_epochs):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for i, data in enumerate(train_loader, 0):
            images, labels = data
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f'Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader):.3f}')

def test_model(model):
    model.eval()
    with torch.no_grad():
        correct = 0
        total = 0
        for data in test_loader:
            images, labels = data
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        print(f'Accuracy on images: {100 * correct / total}')

In [3]:
train_model(model, 15)
#changed to 15 bc of the assignment
test_model(model)

Epoch 1/15, Loss: 2.012
Epoch 2/15, Loss: 1.729
Epoch 3/15, Loss: 1.627
Epoch 4/15, Loss: 1.596
Epoch 5/15, Loss: 1.582
Epoch 6/15, Loss: 1.572
Epoch 7/15, Loss: 1.566
Epoch 8/15, Loss: 1.561
Epoch 9/15, Loss: 1.557
Epoch 10/15, Loss: 1.553
Epoch 11/15, Loss: 1.550
Epoch 12/15, Loss: 1.547
Epoch 13/15, Loss: 1.545
Epoch 14/15, Loss: 1.542
Epoch 15/15, Loss: 1.540
Accuracy on images: 93.43


### Write the interval analysis for the simple model

In [ ]:
## TODO: Write the interval analysis for the simple model
## you can use https://github.com/Zinoex/bound_propagation

def linear_interval(layer, lower, upper):
    #Propagate interval bounds through a linear layer
    weight = layer.weight
    bias = layer.bias

    weight_pos = torch.clamp(weight, min=0)
    weight_neg = torch.clamp(weight, max=0)

    lower_out = F.linear(lower, weight_pos) + F.linear(upper, weight_neg)
    upper_out = F.linear(upper, weight_pos) + F.linear(lower, weight_neg)
    #weight limitations: weight_pos * lower + weight_neg * upper for lower bound
    #weight_pos * upper + weight_neg * lower for upper bound"""
    if bias is not None:
        lower_out = lower_out + bias
        upper_out = upper_out + bias

    return lower_out, upper_out

def interval_relu(lower, upper):
    #Propagate interval bounds through ReLU.
    return torch.clamp(lower, min=0), torch.clamp(upper, min=0)


def interval_forward(model, lower, upper):
    #Propagate input bounds through the network, up to the final linear layer

    net = model[1] 

    #Normalize bounds to match the normalization layer in the model
    lower = (lower - 0.1307) / 0.3081
    upper = (upper - 0.1307) / 0.3081
    # we got these numbers from the Normalize class, which normalizes the input images by subtracting the mean (0.1307) and dividing by the standard deviation (0.3081) of the MNIST dataset.
    # code was taken from the Normalize class implementation
    # also shoutout to the github for the help

    #flatten each image
    lower = lower.reshape(lower.shape[0], -1)
    upper = upper.reshape(upper.shape[0], -1)

    #first linear layer
    lower, upper = linear_interval(net.fc, lower, upper)

    # relu layer
    lower, upper = interval_relu(lower, upper)

    # return lower bounds and upper bounds for the final linear layer
    lower, upper = linear_interval(net.fc2, lower, upper)

    return lower, upper


def margin_lower_bounds(model, lower, upper, labels):
    
    #Compute lower bounds for f_y(x') - f_j(x') for every class j != y
    
    #get the network from the model
    #get interval bounds for the final linear layer
    net = model[1]

    lower = (lower - 0.1307) / 0.3081
    upper = (upper - 0.1307) / 0.3081

    lower = lower.reshape(lower.shape[0], -1)
    upper = upper.reshape(upper.shape[0], -1)

    lower, upper = linear_interval(net.fc, lower, upper)
    lower, upper = interval_relu(lower, upper)
    #same justification as above, we are just propagating the bounds through the network to get the final layer's bounds
    #build an output layer whose outputs are f_y(x') - f_j(x').
    #shape of weight_diff and bias_diff will be (batch_size, num_classes, num_classes)
    weight_diff = (
        net.fc2.weight[labels].unsqueeze(1)
        - net.fc2.weight.unsqueeze(0)
    )
    bias_diff = (
        net.fc2.bias[labels].unsqueeze(1)
        - net.fc2.bias.unsqueeze(0)
    )

    #interval bounds for f_y(x') - f_j(x') are given by:
    weight_pos = torch.clamp(weight_diff, min=0)
    weight_neg = torch.clamp(weight_diff, max=0)

    lower_margins = (
        (weight_pos * lower.unsqueeze(1)).sum(dim=2)
        + (weight_neg * upper.unsqueeze(1)).sum(dim=2)
        + bias_diff
    )

    return lower_margins


def verified_accuracy(model, epsilons):
    #Measure the fraction of test images certified robust at each epsilon.
    model.eval()
    results = {}

    for eps in epsilons:
        verified = 0
        total = 0

        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            
            lower = torch.clamp(images - eps, min=0.0, max=1.0)
            upper = torch.clamp(images + eps, min=0.0, max=1.0)

            with torch.no_grad():
                #lower bounds for f_y(x') - f_j(x')
                margins = margin_lower_bounds(
                    model, lower, upper, labels
                )

                #ignore margins for the true class, since we only care about other classes
                margins[
                    torch.arange(labels.size(0), device=device), labels
                ] = float("inf")

                
                #lower bounds for f_y(x') - f_j(x') >= 0 for all j != y means that the model is certified robust for this image
                certified = (margins >= 0).all(dim=1)

                verified += certified.sum().item()
                total += labels.size(0)

        accuracy = 100.0 * verified / total
        results[eps] = accuracy
        print(f"Epsilon: {eps:.3f} | Verified accuracy: {accuracy:.2f}%")

    return results


epsilons = np.linspace(0.01, 0.1, 10)
results = verified_accuracy(model, epsilons)
#testign! 
#explanations will also be in the report, but this is the code for the interval analysis of the model
#!!!



Epsilon: 0.010 | Verified accuracy: 70.55%
Epsilon: 0.020 | Verified accuracy: 23.68%
Epsilon: 0.030 | Verified accuracy: 4.56%
Epsilon: 0.040 | Verified accuracy: 0.58%
Epsilon: 0.050 | Verified accuracy: 0.02%
Epsilon: 0.060 | Verified accuracy: 0.00%
Epsilon: 0.070 | Verified accuracy: 0.00%
Epsilon: 0.080 | Verified accuracy: 0.00%
Epsilon: 0.090 | Verified accuracy: 0.00%
Epsilon: 0.100 | Verified accuracy: 0.00%
